# Limpieza y Normalización de Datos
## Proyecto UNICEF Argentina — Semana 2

Este notebook documenta el proceso completo de auditoría y limpieza de los 8 datasets
del proyecto. Cada sección detalla qué se transformó, por qué, y cuál fue el resultado.
Las decisiones de limpieza quedan registradas también en `decisions_log.md`.

In [15]:
import pandas as pd
import os

BASE = r"C:\Users\fedeo\Desktop\PROYECTO UNICEF"
RAW = os.path.join(BASE, "data", "raw")
CLEAN = os.path.join(BASE, "data", "clean")

# Matrícula total
df_mat_2022 = pd.read_csv(os.path.join(RAW, "2022 Matricula - agregada(in).csv"), sep=";", encoding="latin-1")
df_mat_2023 = pd.read_csv(os.path.join(RAW, "2023 Matricula - agregada(in).csv"), sep=";", encoding="latin-1")
df_mat_2024 = pd.read_csv(os.path.join(RAW, "2024 Matricula - agregada(in).csv"), sep=";", encoding="latin-1")

# Matrícula por edad
df_edad_2022 = pd.read_csv(os.path.join(RAW, "2022 Matricula por edad - agregada(in).csv"), sep=";", encoding="latin-1")
df_edad_2023 = pd.read_csv(os.path.join(RAW, "2023 Matricula por edad - agregada(in).csv"), sep=";", encoding="latin-1")
df_edad_2024 = pd.read_csv(os.path.join(RAW, "2024 Matricula por edad - agregada(in).csv"), sep=";", encoding="latin-1")

# Abandono
abandono_reciente = pd.read_excel(os.path.join(RAW, "Tasa de Abandono Interanual 2024-2012 según división político-territorial.xlsx"), skiprows=10)
abandono_historico = pd.read_excel(os.path.join(RAW, "Tasa de Abandono Interanual 2016-2003 según estructura nacional homogenea.xlsx"), skiprows=10)

print("Datos cargados correctamente")

Datos cargados correctamente


## 1. Normalización de columnas — Matrícula total

Los datasets de matrícula 2022 y 2023 ya tienen columnas en minúscula.
El dataset 2024 presenta dos columnas con mayúscula inicial (`Departamento`, `Localizacion`)
que se corrigen con rename para mantener consistencia entre los tres años.

In [16]:
# df_mat_2022 y df_mat_2023 ya tienen columnas en minúscula
# df_mat_2024 tiene dos columnas con mayúscula — las corregimos
df_mat_2024 = df_mat_2024.rename(columns={
    "Departamento": "departamento",
    "Localizacion": "localizacion"
})

# Verificación
print("df_mat_2022:", list(df_mat_2022.columns[:4]))
print("df_mat_2023:", list(df_mat_2023.columns[:4]))
print("df_mat_2024:", list(df_mat_2024.columns[:4]))

df_mat_2022: ['provincia', 'departamento', 'sector', 'ambito']
df_mat_2023: ['provincia', 'departamento', 'sector', 'ambito']
df_mat_2024: ['provincia', 'departamento', 'sector', 'ambito']


## 2. Normalización de columnas — Matrícula por edad

Las columnas de estos datasets contienen tildes, ñ y espacios,
lo que puede generar errores al referenciarlas en código.
Se define la función `limpiar_columnas()` que estandariza todo a snake_case sin caracteres especiales
y se aplica a los tres años.

In [17]:
def limpiar_columnas(df):
    df.columns = (
        df.columns
        .str.lower()
        .str.replace("á", "a", regex=False)
        .str.replace("é", "e", regex=False)
        .str.replace("í", "i", regex=False)
        .str.replace("ó", "o", regex=False)
        .str.replace("ú", "u", regex=False)
        .str.replace("ñ", "n", regex=False)
        .str.replace(" ", "_", regex=False)
    )
    return df

df_edad_2022 = limpiar_columnas(df_edad_2022)
df_edad_2023 = limpiar_columnas(df_edad_2023)
df_edad_2024 = limpiar_columnas(df_edad_2024)

# Verificación
print(list(df_edad_2022.columns))

['provincia', 'departamento', 'sector', 'ambito', 'grado', '0anos', '1ano', '2anos', '3anos', '4anos', '5anos', '6anos', '7anos', '8anos', '9anos', '10anos', '11anos', '12anos', '13anos', '14anos', '15anos', '16anos', '17anos', '18anos', '19anos', '20anos', '21anos', '22anos', '23anos', '24anos', '25anos', '26anos', '27anos', '28anos', '29anos', 'menosde18anos', '11anosymenos', '6anosymas', '18anosymas', '25anosymas', '30anosymas', 'de20a24anos', 'de25a29anos']


## 3. Normalización de columnas — Abandono interanual (2012–2024)

El Excel original usa nombres de columna ambiguos (`Unnamed`, `Total`, `Total.1`, `7° Año`, etc.).
Se aplica rename explícito asignando prefijos claros:
`prim_` para nivel primario y `sec_` para nivel secundario.

In [18]:
abandono_reciente = abandono_reciente.rename(columns={
    "Unnamed: 0": "division",
    "Unnamed: 1": "estructura",
    "Total":      "prim_total",
    "1° Año":     "prim_1",
    "2° Año":     "prim_2",
    "3° Año":     "prim_3",
    "4° Año":     "prim_4",
    "5° Año":     "prim_5",
    "6° Año":     "prim_6",
    "7° Año":     "prim_7",
    "Total.1":    "sec_total",
    "7° Año.1":   "sec_7",
    "8° Año":     "sec_8",
    "9° Año":     "sec_9",
    "10° Año":    "sec_10",
    "11° Año":    "sec_11",
    "12° Año":    "sec_12",
})

print(list(abandono_reciente.columns))

['division', 'estructura', 'prim_total', 'prim_1', 'prim_2', 'prim_3', 'prim_4', 'prim_5', 'prim_6', 'prim_7', 'sec_total', 'sec_7', 'sec_8', 'sec_9', 'sec_10', 'sec_11', 'sec_12']


## 4. Normalización de columnas — Abandono interanual (2003–2016)

Similar al dataset anterior, con la diferencia de que la secundaria
se subdivide en ciclo básico (`sec_cb_`) y ciclo orientado (`sec_co_`).
Además, las dos primeras filas eran sub-encabezados del Excel original
y se eliminan antes de continuar.

In [19]:
# Paso 1 — Renombrar columnas
abandono_historico = abandono_historico.rename(columns={
    "División":                                    "division",
    "Primaria (6 años)":                           "prim_total",
    "Unnamed: 2":                                  "prim_1",
    "Unnamed: 3":                                  "prim_2",
    "Unnamed: 4":                                  "prim_3",
    "Unnamed: 5":                                  "prim_4",
    "Unnamed: 6":                                  "prim_5",
    "Unnamed: 7":                                  "prim_6",
    "Secundaria                          (6 años)": "sec_total",
    "Secundaria - Ciclo Básico":                   "sec_cb_total",
    "Unnamed: 10":                                 "sec_7",
    "Unnamed: 11":                                 "sec_8",
    "Unnamed: 12":                                 "sec_9",
    "Secundaria - Ciclo Orientado":                "sec_co_total",
    "Unnamed: 14":                                 "sec_10",
    "Unnamed: 15":                                 "sec_11",
    "Unnamed: 16":                                 "sec_12",
})

# Paso 2 — Eliminar las dos filas que eran sub-encabezados
abandono_historico = abandono_historico.drop(index=[0, 1]).reset_index(drop=True)

# Verificación
print(list(abandono_historico.columns))
print(f"\nShape: {abandono_historico.shape}")
print(abandono_historico.head(2).to_string())

['division', 'prim_total', 'prim_1', 'prim_2', 'prim_3', 'prim_4', 'prim_5', 'prim_6', 'sec_total', 'sec_cb_total', 'sec_7', 'sec_8', 'sec_9', 'sec_co_total', 'sec_10', 'sec_11', 'sec_12']

Shape: (34, 17)
                                             division prim_total    prim_1    prim_2    prim_3    prim_4    prim_5    prim_6 sec_total sec_cb_total     sec_7      sec_8      sec_9 sec_co_total     sec_10    sec_11     sec_12
0                                          Total País   0.685393  0.062613  1.050136  0.911776  0.969154  1.111616 -0.001746  9.327532     7.559821  2.294733  10.174531  10.431338    13.135019  11.656032  8.507842  20.745853
1  Buenos Aires                                         0.311793  -0.65146  0.881044  0.708582  0.552046  0.577692 -0.215944  9.589413     7.680788  3.170301  10.486457   9.704291    14.171057  12.868397  9.670484  21.656545


## 5. Conversión de columnas a numérico — Abandono

Las columnas de los datasets de abandono se importaron como `object` (texto)
debido a que el Excel original mezcla valores numéricos con celdas de encabezado.
Se convierten a `float64` usando `pd.to_numeric(errors='coerce')`:
los valores no convertibles se transforman en `NaN`, haciéndolos visibles para la auditoría siguiente.

In [20]:
# Columnas numéricas en ambos datasets (todas menos las identificadoras)
cols_abandono_reciente = abandono_reciente.columns.drop(["division", "estructura"])
cols_abandono_historico = abandono_historico.columns.drop(["division"])

abandono_reciente[cols_abandono_reciente] = abandono_reciente[cols_abandono_reciente].apply(pd.to_numeric, errors="coerce")
abandono_historico[cols_abandono_historico] = abandono_historico[cols_abandono_historico].apply(pd.to_numeric, errors="coerce")

# Verificación
print(abandono_reciente.dtypes)
print("---")
print(abandono_historico.dtypes)

division       object
estructura     object
prim_total    float64
prim_1        float64
prim_2        float64
prim_3        float64
prim_4        float64
prim_5        float64
prim_6        float64
prim_7        float64
sec_total     float64
sec_7         float64
sec_8         float64
sec_9         float64
sec_10        float64
sec_11        float64
sec_12        float64
dtype: object
---
division         object
prim_total      float64
prim_1          float64
prim_2          float64
prim_3          float64
prim_4          float64
prim_5          float64
prim_6          float64
sec_total       float64
sec_cb_total    float64
sec_7           float64
sec_8           float64
sec_9           float64
sec_co_total    float64
sec_10          float64
sec_11          float64
sec_12          float64
dtype: object


## 6. Detección de nulos

Se auditan los 8 datasets en busca de valores faltantes.
Esta etapa es necesaria antes de cualquier análisis —
los nulos no detectados pueden distorsionar estadísticas y modelos posteriores.

In [21]:
datasets = {
    "df_mat_2022": df_mat_2022,
    "df_mat_2023": df_mat_2023,
    "df_mat_2024": df_mat_2024,
    "df_edad_2022": df_edad_2022,
    "df_edad_2023": df_edad_2023,
    "df_edad_2024": df_edad_2024,
    "abandono_reciente": abandono_reciente,
    "abandono_historico": abandono_historico,
}

for nombre, df in datasets.items():
    nulos = df.isnull().sum()
    nulos_totales = nulos.sum()
    print(f"\n{nombre} — nulos totales: {nulos_totales}")
    if nulos_totales > 0:
        print(nulos[nulos > 0])


df_mat_2022 — nulos totales: 0

df_mat_2023 — nulos totales: 0

df_mat_2024 — nulos totales: 0

df_edad_2022 — nulos totales: 0

df_edad_2023 — nulos totales: 0

df_edad_2024 — nulos totales: 0

abandono_reciente — nulos totales: 204
division       4
estructura     9
prim_total    11
prim_1        11
prim_2        11
prim_3        11
prim_4        11
prim_5        11
prim_6        11
prim_7        25
sec_total     11
sec_7         23
sec_8         11
sec_9         11
sec_10        11
sec_11        11
sec_12        11
dtype: int64

abandono_historico — nulos totales: 116
division        4
prim_total      7
prim_1          7
prim_2          7
prim_3          7
prim_4          7
prim_5          7
prim_6          7
sec_total       7
sec_cb_total    7
sec_7           7
sec_8           7
sec_9           7
sec_co_total    7
sec_10          7
sec_11          7
sec_12          7
dtype: int64


## 7. Inspección de nulos — abandono_reciente

Se identifican tres tipos de nulos en este dataset:
- **Estructurales esperados:** provincias con estructura `6-6` no tienen `prim_7`; con `7-5` no tienen `sec_7`. Son "no aplica", no datos faltantes.
- **Fila sin estructura:** `Total País` no tiene valor en `estructura`. Correcto.
- **Filas basura:** pie de página del Excel original — texto descriptivo y fecha de realización. Se eliminan en la sección siguiente.

In [22]:
print(abandono_reciente[abandono_reciente.isnull().any(axis=1)].to_string())

                                                                                               division   estructura  prim_total    prim_1    prim_2    prim_3    prim_4    prim_5    prim_6    prim_7  sec_total     sec_7     sec_8      sec_9     sec_10     sec_11     sec_12
0                                                                                            Total País          NaN    0.516238 -0.147195  0.919616  0.882836  0.629229  0.778037  0.071039  0.236353   8.047195  2.171372  4.197437   5.524445   9.761971   8.192727  19.990978
1                                                    Buenos Aires                                                6-6    0.405685 -0.691481  1.179272  0.894652  0.508399  0.661824 -0.232935       NaN   6.688718  1.058992  2.846020   2.698364   9.784367   8.523543  18.661079
2                                                                 Conurbano                                      6-6    0.589701 -0.547618  1.425830  1.210142  0.691566  0.832068

## 8. Eliminación de filas basura — abandono_reciente

Se eliminan las filas del pie de página del Excel original
(descripciones de estructura educativa y fecha de realización).
El criterio es: filas donde **todas** las columnas numéricas son `NaN`.
Shape esperado post-limpieza: 27 filas × 17 columnas.

In [23]:
cols_numericas = abandono_reciente.columns.drop(["division", "estructura"])
abandono_reciente = abandono_reciente.dropna(subset=cols_numericas, how="all").reset_index(drop=True)

print(f"Shape final: {abandono_reciente.shape}")
print(abandono_reciente.tail(3).to_string())

Shape final: (27, 17)
                                              division estructura  prim_total    prim_1    prim_2    prim_3    prim_4    prim_5    prim_6    prim_7  sec_total     sec_7     sec_8     sec_9     sec_10    sec_11     sec_12
24  Santiago Del Estero                                       7-5    0.901037 -0.247916  0.897247  2.122249  1.137168  2.072220  1.257473 -1.074506   9.100119       NaN  8.703357  8.546103  10.705580  7.458191  10.421989
25  Tierra Del Fuego                                          6-6    1.098026  0.945709  1.253133  1.511740  0.793913  1.163985  0.904033       NaN   6.196243  1.256198  2.498378  2.147341   5.145339  3.201970  24.787273
26  Tucumán                                                   6-6    0.339305  0.473916  0.111107  0.399045  0.416441  0.643389 -0.013967       NaN   7.096997  2.549284  5.584654  5.926763   6.788554  6.450333  18.763594


## 9. Inspección de nulos — abandono_historico

Se auditan los valores nulos del dataset de abandono histórico (2003–2016)
para determinar si son estructurales, heredados del Excel original,
o producto de errores en la carga o conversión a numérico.
Este dataset ya tuvo dos intervenciones previas: eliminación de filas
de sub-encabezado (`drop(index=[0,1])`) y conversión de columnas a `float64`.
Se esperan pocos nulos, pero hay que confirmarlo antes de cerrar la limpieza.

In [24]:
# ¿Cuántos nulos tiene en total y dónde?
print("Shape:", abandono_historico.shape)
print("\nNulos por columna:")
print(abandono_historico.isnull().sum())

Shape: (34, 17)

Nulos por columna:
division        4
prim_total      7
prim_1          7
prim_2          7
prim_3          7
prim_4          7
prim_5          7
prim_6          7
sec_total       7
sec_cb_total    7
sec_7           7
sec_8           7
sec_9           7
sec_co_total    7
sec_10          7
sec_11          7
sec_12          7
dtype: int64


## 10. Eliminación de filas basura — abandono_historico

Se eliminan las filas del pie de página del Excel original
(definición metodológica, cita de fuente, fecha de elaboración y filas vacías de separación).
El criterio es el mismo que en `abandono_reciente`: filas donde **todas**
las columnas numéricas son `NaN`.
Shape esperado post-limpieza: 27 filas × 17 columnas.

In [25]:
cols_numericas = abandono_historico.columns.drop('division')
abandono_historico = abandono_historico.dropna(subset=cols_numericas, how='all').reset_index(drop=True)
print("Shape final:", abandono_historico.shape)
print("\nNulos restantes por columna:")
print(abandono_historico.isnull().sum())

Shape final: (27, 17)

Nulos restantes por columna:
division        0
prim_total      0
prim_1          0
prim_2          0
prim_3          0
prim_4          0
prim_5          0
prim_6          0
sec_total       0
sec_cb_total    0
sec_7           0
sec_8           0
sec_9           0
sec_co_total    0
sec_10          0
sec_11          0
sec_12          0
dtype: int64


## 11. Exportación — Datasets limpios a data/clean/

Los 8 datasets limpios se exportan a `data/clean/` como archivos CSV en encoding UTF-8.
Se usa `index=False` para evitar guardar el índice de pandas como columna extra.
A partir de este punto, todo el análisis trabaja sobre los archivos de `data/clean/`,
nunca sobre los originales de `data/raw/`.

In [26]:
os.makedirs(CLEAN, exist_ok=True)

df_mat_2022.to_csv(os.path.join(CLEAN, "matricula_total_2022.csv"), index=False, encoding="utf-8")
df_mat_2023.to_csv(os.path.join(CLEAN, "matricula_total_2023.csv"), index=False, encoding="utf-8")
df_mat_2024.to_csv(os.path.join(CLEAN, "matricula_total_2024.csv"), index=False, encoding="utf-8")

df_edad_2022.to_csv(os.path.join(CLEAN, "matricula_edad_2022.csv"), index=False, encoding="utf-8")
df_edad_2023.to_csv(os.path.join(CLEAN, "matricula_edad_2023.csv"), index=False, encoding="utf-8")
df_edad_2024.to_csv(os.path.join(CLEAN, "matricula_edad_2024.csv"), index=False, encoding="utf-8")

abandono_reciente.to_csv(os.path.join(CLEAN, "abandono_reciente.csv"),  index=False, encoding="utf-8")
abandono_historico.to_csv(os.path.join(CLEAN, "abandono_historico.csv"), index=False, encoding="utf-8")

print("✓ 8 datasets exportados a data/clean/")
print(f"   Ubicación: {CLEAN}")

✓ 8 datasets exportados a data/clean/
   Ubicación: C:\Users\fedeo\Desktop\PROYECTO UNICEF\data\clean
